In [1]:
import torch

In [2]:
!pip install transformers datasets evaluate accelerate scikit-learn -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [3]:


from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import numpy as np



print("Loading dataset...")
dataset = load_dataset("stanfordnlp/imdb")

# Peek at one example so you can see what the raw data looks like
print("Example entry:", dataset["train"][0])



model_name = "distilbert-base-uncased"
print(f"Loading tokenizer and model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


def tokenize_function(examples):
    # truncation=True cuts off very long reviews so they fit the model's limit
    # padding="max_length" makes every example the same length (required for
    # batching multiple examples together efficiently)
    return tokenizer(examples["text"], truncation=True, padding="max_length")


print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Using a small subset for your first run so training finishes quickly.
# Once this works end-to-end, you can increase these numbers (or remove
# .select(...) entirely to use the full dataset) for a stronger model.
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(500))


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # The model outputs a score for each class; we pick whichever is highest
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",            # check performance after each full pass
    save_strategy="epoch",
    learning_rate=2e-5,               # size of each adjustment step (small = safer)
    per_device_train_batch_size=8,    # how many examples processed at once
    per_device_eval_batch_size=8,
    num_train_epochs=2,               # how many full passes through the training data
    weight_decay=0.01,                # helps prevent overfitting
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()


print("Evaluating final model...")
results = trainer.evaluate()
print("Final results:", results)


trainer.save_model("./my-sentiment-model")
tokenizer.save_pretrained("./my-sentiment-model")
print("Done!.")

Loading dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

Evaluating final model...


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.183838,0.365678,2,0.888000,0.889764


Final results: {'eval_loss': 0.3656783401966095, 'eval_accuracy': 0.888, 'eval_f1': 0.889763779527559}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done!.


In [4]:
# Try the fine-tuned model on new text it's never seen
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # move inputs to same device as model
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.nn.functional.softmax(logits, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_class].item()
    label = "Positive" if predicted_class == 1 else "Negative"
    return label, confidence

# Test 1: A normal movie review (in-domain — should work well)
text1 = "This movie was a complete waste of time, terribly acted and boring."
label1, conf1 = predict_sentiment(text1)
print(f"Text: {text1}\nPrediction: {label1} (confidence: {conf1:.2%})\n")

# Test 2: A restaurant review instead of movie (different domain)
text2 = "The food was cold and the service was painfully slow."
label2, conf2 = predict_sentiment(text2)
print(f"Text: {text2}\nPrediction: {label2} (confidence: {conf2:.2%})\n")

# Test 3: Something totally unrelated to sentiment at all
text3 = "The mitochondria is the powerhouse of the cell."
label3, conf3 = predict_sentiment(text3)
print(f"Text: {text3}\nPrediction: {label3} (confidence: {conf3:.2%})\n")

# Test 4: Bangla text (completely different language)
text4 = "এই সিনেমাটা অসাধারণ ছিল।"  # "This movie was amazing"
label4, conf4 = predict_sentiment(text4)
print(f"Text: {text4}\nPrediction: {label4} (confidence: {conf4:.2%})\n")

Text: This movie was a complete waste of time, terribly acted and boring.
Prediction: Negative (confidence: 98.73%)

Text: The food was cold and the service was painfully slow.
Prediction: Negative (confidence: 98.15%)

Text: The mitochondria is the powerhouse of the cell.
Prediction: Positive (confidence: 59.09%)

Text: এই সিনেমাটা অসাধারণ ছিল।
Prediction: Positive (confidence: 65.60%)



In [7]:
model.push_to_hub("review-recognizer")
tokenizer.push_to_hub("review-recognizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v0j9a1e/model.safetensors:   6%|5         | 15.9MB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/authoy6788/review-recognizer/commit/8f40073d73aaa176ba3b14b68e065ec8f6a814d9', commit_message='Upload tokenizer', commit_description='', oid='8f40073d73aaa176ba3b14b68e065ec8f6a814d9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/authoy6788/review-recognizer', endpoint='https://huggingface.co', repo_type='model', repo_id='authoy6788/review-recognizer'), pr_revision=None, pr_num=None)